# DepChainTagger demo

This notebook shows how to configure `DepChainTagger`, how to define a `PathPattern`, and how to run the tagger on a sample text. It also explains the most important parts of the output so you can adapt the example to your own texts.


## What this notebook covers

1. The constructor parameters exposed by `DepChainTagger`.
2. How to build a simple dependency-chain pattern with `NodeConstraint`, `EdgeConstraint`, and `PathPattern`.
3. How to prepare an `estnltk.Text` object with syntax layers.
4. How to run the tagger and inspect the resulting matches.

The sample code uses a broad two-node pattern so the mechanics are easy to see. The exact number of matches depends on the quality of the syntax parse and the Stanza model version used in your environment.


In [ ]:
from pprint import pprint

import estnltk

from scripts.DepChainTagger import (
    ConditionMode,
    DepChainTagger,
    DepTaggerOrchestrator,
    DirectionMode,
    EdgeConstraint,
    NodeConstraint,
    PathPattern,
    ValueCondition,
)


## Tagger parameters

The current `DepChainTagger` constructor focuses on output control and match filtering. The tagger always reads from the `stanza_syntax` and `sentences` layers internally, and it writes a relation layer named `dep_chains` by default.

| Parameter                   | What it controls                                                                       |
| --------------------------- | -------------------------------------------------------------------------------------- |
| `patterns`                  | A tuple of `PathPattern` objects that define what the tagger should match.             |
| `output_layer`              | Name of the relation layer created by the tagger.                                      |
| `output_attributes`         | Extra attributes written to the output layer. If omitted, the package default is used. |
| `sentence_match_dedup_mode` | Deduplication inside one sentence: `none`, `exact`, or `role_based`.                   |
| `max_matches_per_sentence`  | Maximum number of matches to keep per sentence.                                        |
| `allow_role_node_overlap`   | Whether the same node may fill more than one role in a pattern.                        |
| `global_dedup_mode`         | Deduplication across the whole text: `none`, `exact`, or `role_based`.                 |
| `max_total_matches`         | Upper limit for all matches across the entire text.                                    |

The next cells print the instantiated values so you can see exactly which configuration is active.


In [ ]:
def build_demo_pattern() -> PathPattern:
    """Build a small dependency-chain pattern for the demo notebook.

    The example uses two roles, `parent` and `child`, and keeps the
    attribute checks intentionally broad so the notebook demonstrates the
    mechanics of pattern definition rather than a narrow linguistic rule.

    Returns:
        PathPattern: A reusable pattern object for the tagger.
    """

    parent_node = NodeConstraint(
        role="parent",
        attribute_conditions={"upostag": ValueCondition(ConditionMode.WILDCARD)},
    )
    child_node = NodeConstraint(
        role="child",
        attribute_conditions={"upostag": ValueCondition(ConditionMode.WILDCARD)},
    )
    edge_constraint = EdgeConstraint(
        direction=DirectionMode.UP,
        attribute_conditions={"deprel": ValueCondition(ConditionMode.WILDCARD)},
        min_hops=1,
        max_hops=1,
    )

    return PathPattern(
        name="demo_parent_child",
        node_steps=(parent_node, child_node),
        edge_steps=(edge_constraint,),
        anchor_role="child",
        emit_roles=("parent", "child"),
    )


demo_pattern = build_demo_pattern()
print(demo_pattern.describe())


## Prepare a sample text

The tagger expects a text object with sentence segmentation and a syntax layer. The code below uses the same sample sentence family as the test suite so the example stays close to the repository's own fixtures.

If `estnltk_neural` or the Stanza syntax tagger is not available in your environment, the notebook will still explain the workflow, but the final tagging step will not run.


In [ ]:
sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text = estnltk.Text(sample_text)
text.tag_layer("morph_extended")

syntax_ready = False
try:
    from estnltk_neural.taggers import StanzaSyntaxTagger

    stanza_tagger = StanzaSyntaxTagger(
        input_type="morph_analysis",
        input_morph_layer="morph_analysis",
    )
    stanza_tagger.tag(text)
    syntax_ready = True
    print("Syntax layer created successfully.")
    print("Sentences detected:", len(text.sentences))
except Exception as exc:
    print("StanzaSyntaxTagger is not available in this environment.")
    print("Tagging will be explained, but not executed here.")
    print(f"Reason: {exc}")


## Run `DepChainTagger`

The next cell constructs the tagger with explicit values for the most important constructor arguments and then applies it to the text.

If the syntax layer was built successfully, the call to `tagger.tag(text)` adds the output layer to the `Text` object. You should then be able to inspect the new layer by name.


In [ ]:
tagger = DepChainTagger(
    patterns=(demo_pattern,),
    output_layer="dep_chains",
    sentence_match_dedup_mode="role_based",
    max_matches_per_sentence=100,
    allow_role_node_overlap=False,
    global_dedup_mode="none",
    max_total_matches=1000,
)

print("Active DepChainTagger configuration:")
print("  output_layer:", tagger.output_layer)
print("  output_span_names:", tagger.output_span_names)
print("  output_attributes:", tagger.output_attributes)
print("  conf_param:", tagger.conf_param)

if syntax_ready:
    tagged_text = tagger.tag(text)
    print("")
    print("Tagging completed. The output layer was added to the text object.")
    print("Available layer names:", list(tagged_text.layers))
else:
    print("")
    print("Tagging was skipped because the syntax layer is missing.")


## Inspect the output rows

`DepChainTagger` creates a relation layer, but it is often easier to understand the results through the decorated rows that the orchestrator produces. Each row represents one successful match and includes the pattern name, sentence context, the matched text, role-to-token mapping, role-to-text mapping, and character spans.

The meaning of the main fields is:

- `pattern_name`: which `PathPattern` matched.
- `sentence_index` and `sentence_span`: where the match came from.
- `matched_text`: the human-readable text covered by the emitted roles.
- `role_to_token_id`: token ids assigned to each role.
- `role_to_text`: the surface forms for each role.
- `role_to_span`: character spans for each role.
- `traversed_edges`: how the matcher walked the dependency graph.

The exact values depend on the parser output, but the structure stays the same.


In [ ]:
if syntax_ready:
    sentence_layers = [sentence.stanza_syntax for sentence in text.sentences]
    sentence_spans = [(sentence.start, sentence.end) for sentence in text.sentences]

    orchestrator = DepTaggerOrchestrator(
        patterns=(demo_pattern,),
        sentence_match_dedup_mode="role_based",
        global_dedup_mode="none",
        max_total_matches=1000,
    )

    rows = orchestrator.tag_and_decorate_sentence_layers(
        sentence_syntax_layers=sentence_layers,
        sentence_spans=sentence_spans,
    )

    print(f"Number of decorated matches: {len(rows)}")
    if rows:
        pprint(rows[0])
else:
    print("No syntax layer is available, so there are no output rows to inspect.")


## How to read the result

If the notebook runs successfully, the first printed row should look like a dictionary. That is deliberate: the decorated format is designed to be easy to inspect, serialise, or export to another tool.

When the output contains several matches, it usually means the pattern is broad. To make it more specific, tighten the `NodeConstraint` values, add more edge constraints, or reduce the allowed hop range.

Good next steps:

1. Replace the wildcard `ValueCondition` values with exact part-of-speech or dependency-label checks.
2. Add more roles to the pattern when you want to trace a longer chain.
3. Change `sentence_match_dedup_mode` and `global_dedup_mode` if you want fewer repeated matches.
